# LoRA vs Full Fine-Tuning on RoBERTa-base

This notebook clones the project repo, installs dependencies, and runs all six
experiments:

| Mode | Dataset | Description |
|---|---|---|
| baseline | SST-2 | Zero-shot evaluation (no training) |
| finetune | SST-2 | Full fine-tuning |
| lora | SST-2 | LoRA fine-tuning |
| baseline | MNLI | Zero-shot evaluation (no training) |
| finetune | MNLI | Full fine-tuning |
| lora | MNLI | LoRA fine-tuning |

Results are collected and printed as a comparison table at the end.

## 1. Setup

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
REPO_URL      = "https://github.com/da-luce/cs5782_final_project.git"
BRANCH        = "baseline"   # must be "main" for the final submission
TRAIN_SAMPLES = 5000
VAL_SAMPLES   = 500
EPOCHS        = 5
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
REPO_DIR = REPO_URL.split("/")[-1].replace(".git", "")

!git clone -b {BRANCH} {REPO_URL}
%cd {REPO_DIR}

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. SST-2 Experiments

#### SST-2 Dataset

SST-2 is a sentiment analysis dataset, and a component of the GLUE benchmark. Each sentence is manually annotated with a binary label (positive or negative) to indicate the sentiment expressed by the reviewer.

This is a relatively **simple** task.

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset sst2 \
    --val_samples {VAL_SAMPLES}

In [ ]:
!python code/train.py \
    --mode finetune \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 3. MNLI Experiments

#### MNLI Dataset

MNLI (Multi-Genre Natural Language Inference) is a large dataset and a key component of the GLUE benchmark. Each entry consists of a sentence pair--a premise and a hypothesis--manually annotated with a label of entailment, contradiction, or neutral.

MNLI includes a diverse range of written and spoken English across ten different genres. This makes it a significantly more **complex** task that tests a model's ability to perform logical reasoning and generalization.

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset mnli \
    --val_samples {VAL_SAMPLES}

In [ ]:
!python code/train.py \
    --mode finetune \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 4. Results Comparison

In [ ]:
import json
import os

experiments = [
    ("baseline", "sst2"),
    ("finetune", "sst2"),
    ("lora",     "sst2"),
    ("baseline", "mnli"),
    ("finetune", "mnli"),
    ("lora",     "mnli"),
]

rows = []
for mode, dataset in experiments:
    path = f"results/{mode}_{dataset}.json"
    if not os.path.exists(path):
        print(f"Missing results for {mode}/{dataset} — skipping")
        continue

    with open(path) as f:
        info = json.load(f)

    accuracy = info["eval"]["results"].get("eval_accuracy", float("nan"))
    trainable = info["model"]["trainable"]
    trainable_pct = info["model"]["trainable_pct"]
    total = info["model"]["total"]

    # Calculate total time from the segmented timers
    train_time = info["training"]["time_sec"]
    eval_time = info["eval"]["time_sec"]
    time_min = (train_time + eval_time) / 60

    rows.append((mode, dataset, accuracy, trainable, trainable_pct, total, time_min))

header = f"{'Mode':<12} {'Dataset':<8} {'Accuracy':>10} {'Trainable params':>18} {'Trainable %':>12} {'Total params':>14} {'Time (min)':>12}"
sep = "-" * len(header)
print(header)
print(sep)
for mode, dataset, acc, tr, tr_pct, tot, t in rows:
    print(f"{mode:<12} {dataset:<8} {acc:>10.4f} {tr:>18,} {tr_pct:>11.4f}% {tot:>14,} {t:>11.1f}m")

In [ ]:
# Print raw JSON for each completed experiment so results are preserved in notebook output
import glob
for path in sorted(glob.glob("results/*.json")):
    print(f"\n{'='*60}\n{path}\n{'='*60}")
    with open(path) as f:
        print(f.read())

In [ ]:
# Download result JSONs to your local machine
from google.colab import files
import glob

for path in sorted(glob.glob("results/*.json")):
    files.download(path)